This notebook does not run in production. 

It was used to fine-tune the Pydantic classes when integrating data source 2, exporting the generated JSONs to a file and then comapring them with the ground truth of the dataset in order to validate. 

Nevertheless I decided to add this to the GitHub repo as a showcase of the model preparation process.

In [1]:
%uv pip install docling
%uv pip install qwen-vl-utils
%uv pip install pymupdf

Using Python 3.12.6 environment at: /usr/local
Resolved 109 packages in 2.25s
Building pylatexenc==2.10
Building pylatexenc==2.10
Building antlr4-python3-runtime==4.9.3
Building pylatexenc==2.10
Building antlr4-python3-runtime==4.9.3
⠙ Preparing packages... (0/39)
Building pylatexenc==2.10
Building antlr4-python3-runtime==4.9.3
⠙ Preparing packages... (0/39)
Building pylatexenc==2.10
Building antlr4-python3-runtime==4.9.3
⠙ Preparing packages... (0/39)
jsonref    ------------------------------     0 B/9.20 KiB
Building pylatexenc==2.10
Building antlr4-python3-runtime==4.9.3
⠙ Preparing packages... (0/39)
jsonref    ------------------------------ 9.20 KiB/9.20 KiB
Building pylatexenc==2.10
Building antlr4-python3-runtime==4.9.3
⠙ Preparing packages... (0/39)
jsonref    ------------------------------ 9.20 KiB/9.20 KiB
et-xmlfile ------------------------------ 14.91 KiB/17.64 KiB
Building pylatexenc==2.10
Building antlr4-python3-runtime==4.9.3
⠙ Preparing packages... (0/39)
jsonref    ---

In [2]:
from PIL import Image
import io
from IPython.display import display, JSON
from pydantic import BaseModel, Field, AliasChoices
from rich import print
import polars as pl
from typing import Optional
import json
from docling.datamodel.document import DocumentStream
from docling.datamodel.base_models import InputFormat
from docling.document_extractor import (
    DocumentExtractor, ExtractionFormatOption,
    ExtractionVlmPipeline, ImageDocumentBackend,
    PyPdfiumDocumentBackend, InputFormat
)
from docling.document_converter import PdfFormatOption, ImageFormatOption, DocumentConverter
import torch
import gc
import tempfile, os
from pathlib import Path
import fitz

In [3]:
#Clearing cache of graphics card
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")

VRAM free: 21.8 GB

In [4]:
#Enabling GPU acceleration for faster model inference

from docling.datamodel.accelerator_options import AcceleratorOptions, AcceleratorDevice
from docling.datamodel.pipeline_options import VlmExtractionPipelineOptions
from docling.document_extractor import DocumentExtractor

acc_options = AcceleratorOptions(device=AcceleratorDevice.CUDA)
pipeline_options = VlmExtractionPipelineOptions(accelerator_options=acc_options)

In [7]:
# This needs to be de-allocated at the end to avoid memory leaks (because it's VRAM and not regular RAM), although it's not absolutely necessary if it runs on Modal, it's a good safety guard.
extractor = DocumentExtractor(
    allowed_formats=[InputFormat.IMAGE],
    extraction_format_options={
        InputFormat.IMAGE: ExtractionFormatOption(
            pipeline_cls=ExtractionVlmPipeline,
            pipeline_options=pipeline_options,
            backend=ImageDocumentBackend,
        )
    }
)

In [8]:
class InvoiceHeader_DS2(BaseModel):
    invoice_date: Optional[str] = Field(
        default=None, 
        description="The invoice or order date",
        examples=["Date", "Date:"],
        validation_alias=AliasChoices("Date", "Order date")
    )
    client: Optional[str] = Field(
        default=None, 
        description="The billed client or customer name",
        examples=["Bill To:", "Bill To:"],
        validation_alias=AliasChoices("Bill To:", "Bill To", "Customer Name", "Buyer")
    )
    client_address: Optional[str] = Field(
        default=None, 
        description="The shipping or delivery address",
        examples=["Ship To:", "Ship To"],
        validation_alias=AliasChoices("Ship To:", "Address", "Ship To")
    )

In [9]:
class InvoiceItem_DS2(BaseModel):
    item_desc: Optional[str] = Field(
        default=None, 
        description="The item or product description",
        examples=["Item"],
        validation_alias=AliasChoices("Item", "Item description", "Item name", "Product name")
    )
    item_qty: Optional[str] = Field(
        default=None, 
        description="The quantity ordered for this product",
        examples=["Quantity"],
        validation_alias=AliasChoices("Quantity", "Item quantity")
    )
    item_rate: Optional[str] = Field(
        default=None,
        description="The unit price or rate",
        examples=["Rate"],
        validation_alias=AliasChoices("Rate", "Price", "Item net price", "Unit cost")
    )

In [10]:
class InvoiceSummary_DS2(BaseModel):
    order_id: Optional[str] = Field(
        default=None,
        description="The order or invoice ID/number",
        examples=["Order ID:", "Order ID"],
        validation_alias=AliasChoices("Order ID", "order")
    )
    discount: Optional[str] = Field(
        default=None, 
        description="Discount amount applied",
        examples=["Discount"],
        validation_alias=AliasChoices("Discount")
    )
    shipping: Optional[str] = Field(
        default=None, 
        description="Shipping fee",
        examples=["Shipping:", "Shipping"],
        validation_alias=AliasChoices("Shipping", "Shipping fee")
    )

In [11]:
class InvoiceDocument_DS2(BaseModel):
    header: InvoiceHeader_DS2
    items: list[InvoiceItem_DS2]
    summary: InvoiceSummary_DS2

In [12]:
def pdf_to_image_bytes_list(pdf_bytes: bytes, dpi: int = 150) -> list[bytes]:
    # Convert each page of a PDF to JPEG bytes.
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    pages_as_images = []
    for page in doc:
        mat = fitz.Matrix(dpi / 72, dpi / 72)  # 72 is PDF's base DPI
        pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
        pages_as_images.append(pix.tobytes("jpeg"))
    doc.close()
    return pages_as_images

In [13]:
def img_bytes_to_json(img_bytes: bytes, pydantic_model):
    # Write to a temp file and delete right after to make it easier for the extractor to parse
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as f:
        f.write(img_bytes)
        tmp_path = Path(f.name)
    try:
        result = extractor.extract(
            source=tmp_path,
            template=pydantic_model,   #The pydantic model that defines the expected JSON structure
            raises_on_error=True,
        )
    finally:
        os.unlink(tmp_path)

    raw = result.pages[0]
    print("extracted_data type:", type(raw.extracted_data))
    print("extracted_data:", raw.extracted_data)
    print("raw_text:", raw.raw_text)

    return json.loads(result.json())["pages"][0]["extracted_data"]

In [14]:
fileList = os.listdir('/root')
pdfPaths = [f'/root/{f}' for f in fileList if f.endswith('.pdf')]
pdfPaths.sort()

In [15]:
print(pdfPaths)

[
    '/root/invoice_Aaron Bergman_36258.pdf',
    '/root/invoice_Aaron Bergman_36259.pdf',
    '/root/invoice_Aaron Bergman_36260.pdf',
    '/root/invoice_Aaron Bergman_39519.pdf',
    '/root/invoice_Aaron Hawkins_36651.pdf',
    '/root/invoice_Aaron Hawkins_36652.pdf',
    '/root/invoice_Aaron Hawkins_37425.pdf',
    '/root/invoice_Aaron Hawkins_38460.pdf',
    '/root/invoice_Aaron Hawkins_38461.pdf',
    '/root/invoice_Aaron Hawkins_40100.pdf',
    '/root/invoice_Aaron Hawkins_40101.pdf',
    '/root/invoice_Aaron Hawkins_47905.pdf',
    '/root/invoice_Aaron Hawkins_4820.pdf',
    '/root/invoice_Aaron Hawkins_49674.pdf',
    '/root/invoice_Aaron Hawkins_6817.pdf',
    '/root/invoice_Aaron Smayling_15978.pdf',
    '/root/invoice_Aaron Smayling_35876.pdf',
    '/root/invoice_Adam Bellavance_21617.pdf',
    '/root/invoice_Adam Hart_16384.pdf',
    '/root/invoice_Adam Hart_30118.pdf',
    '/root/invoice_Adam Hart_30845.pdf',
    '/root/invoice_Adam Hart_36279.pdf',
    '/root/invoice_Adam Shillingsburg_12471.pdf',
    '/root/invoice_Adam Shillingsburg_40245.pdf',
    '/root/invoice_Adam Shillingsburg_40952.pdf',
    '/root/invoice_Adam Shillingsburg_40953.pdf',
    '/root/invoice_Adam Shillingsburg_40954.pdf',
    '/root/invoice_Adam Shillingsburg_40955.pdf',
    '/root/invoice_Adrian Barton_25445.pdf',
    '/root/invoice_Adrian Barton_35579.pdf',
    '/root/invoice_Adrian Barton_35580.pdf',
    '/root/invoice_Adrian Hane_33399.pdf',
    '/root/invoice_Aimee Bixby_39793.pdf',
    '/root/invoice_Aimee Bixby_39794.pdf',
    '/root/invoice_Aimee Bixby_39795.pdf',
    '/root/invoice_Aimee Bixby_39796.pdf',
    '/root/invoice_Aimee Bixby_39797.pdf',
    '/root/invoice_Alan Barnes_36600.pdf',
    '/root/invoice_Alan Dominguez_1626.pdf',
    '/root/invoice_Alan Dominguez_31421.pdf',
    '/root/invoice_Alan Dominguez_41032.pdf',
    '/root/invoice_Alan Haines_22343.pdf',
    '/root/invoice_Alan Haines_29721.pdf',
    '/root/invoice_Alan Haines_36551.pdf',
    '/root/invoice_Alan Haines_36552.pdf',
    '/root/invoice_Alan Haines_42998.pdf',
    '/root/invoice_Alan Haines_5873.pdf',
    '/root/invoice_Alan Hwang_14266.pdf',
    '/root/invoice_Alan Hwang_14267.pdf',
    '/root/invoice_Alan Hwang_14269.pdf',
    '/root/invoice_Alan Hwang_28262.pdf',
    '/root/invoice_Alan Hwang_35712.pdf',
    '/root/invoice_Alan Hwang_35713.pdf',
    '/root/invoice_Alan Schoenberger_33272.pdf',
    '/root/invoice_Alan Schoenberger_33273.pdf',
    '/root/invoice_Alan Schoenberger_33274.pdf',
    '/root/invoice_Alan Shonely_31899.pdf',
    '/root/invoice_Alan Shonely_31900.pdf',
    '/root/invoice_Alan Shonely_31901.pdf',
    '/root/invoice_Alan Shonely_31902.pdf',
    '/root/invoice_Alan Shonely_31903.pdf',
    '/root/invoice_Alan Shonely_37511.pdf',
    '/root/invoice_Alan Shonely_43048.pdf',
    '/root/invoice_Alejandro Ballentine_40780.pdf',
    '/root/invoice_Alejandro Grove_24518.pdf',
    '/root/invoice_Alejandro Grove_38707.pdf',
    '/root/invoice_Alejandro Savely_20581.pdf',
    '/root/invoice_Alejandro Savely_20583.pdf',
    '/root/invoice_Alejandro Savely_41450.pdf',
    '/root/invoice_Aleksandra Gannaway_33911.pdf',
    '/root/invoice_Aleksandra Gannaway_33912.pdf',
    '/root/invoice_Alex Avila_33526.pdf',
    '/root/invoice_Alex Avila_33527.pdf',
    '/root/invoice_Alex Avila_38765.pdf',
    '/root/invoice_Alex Avila_38766.pdf',
    '/root/invoice_Alex Grayson_18794.pdf',
    '/root/invoice_Alex Russell_28157.pdf',
    '/root/invoice_Alex Russell_38655.pdf',
    '/root/invoice_Alex Russell_38656.pdf',
    '/root/invoice_Alice McCarthy_10670.pdf',
    '/root/invoice_Alice McCarthy_36714.pdf',
    '/root/invoice_Alice McCarthy_8673.pdf',
    '/root/invoice_Allen Armold_32104.pdf',
    '/root/invoice_Allen Armold_32105.pdf',
    '/root/invoice_Allen Armold_32469.pdf',
    '/root/invoice_Allen Armold_34304.pdf',
    '/root/invoice_Allen Goldenen_36282.pdf',
    '/root/invoice_Allen Goldenen_39139.pdf',
    '/root/invoice_Allen Goldenen_39140.pdf'

In [47]:
# with open("/root/invoice_Maria Zettner_24429.pdf", "rb") as f:
#     pdf_file_1 = f.read()

# pdf_pages_file_1 = pdf_to_image_bytes_list(pdf_file_1)
# img_bytes = pdf_pages_file_1[0]

In [48]:
result = img_bytes_to_json(img_bytes, InvoiceDocument_DS2)

extracted_data type: <class 'dict'>

extracted_data:
{
    'header': {
        'invoice_date': '2013-03-07',
        'client': 'Maria Zettner',
        'client_address': 'Gold Coast, Queensland, Australia'
    },
    'items': [{'item_desc': 'Hon Rocking Chair, Black', 'item_qty': 4, 'item_rate': 461.48}],
    'summary': {'order_id': 'IN-2013-MZ173357-41340', 'discount': 184.59, 'shipping': 109.26}
}

raw_text: {"header": {"invoice_date": "2013-03-07", "client": "Maria Zettner", "client_address": "Gold Coast, 
Queensland, Australia"}, "items": [{"item_desc": "Hon Rocking Chair, Black", "item_qty": 4, "item_rate": 461.48}], 
"summary": {"order_id": "IN-2013-MZ173357-41340", "discount": 184.59, "shipping": 109.26}}

/tmp/ipykernel_103/2335236361.py:20: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  return json.loads(result.json())["pages"][0]["extracted_data"]


In [28]:
print(result)

None

In [16]:
def prettyfy_extraction_result(result):
    return json.dumps(json.loads(result.json())['pages'][0]['extracted_data'], indent=2)

In [17]:
def parse_invoice_bytes(img_bytes: bytes):
    # Save bytes to temp file with correct extension
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as f:
        f.write(img_bytes)
        tmp_path = Path(f.name)

    try:
        result = extractor.extract(
            source=tmp_path,
            template=InvoiceDocument_DS2,
            raises_on_error=True,
        )
        #print(result)
    finally:
        os.unlink(tmp_path)  # clean up

    return prettyfy_extraction_result(result)

In [ ]:
#result = parse_invoice_bytes(img_bytes)

Here we dump the first 70 invoices into a .txt file in order to be compared against the ground truth, to asses the accuracy of the model

In [19]:
with open("first_70_invoices.txt", "w+") as f:
    f.write("[")
    for i in range(70):
        curr_pdf_path = pdfPaths[i]
        with open(curr_pdf_path, "rb") as f_pdf:
            curr_pdf_file = f_pdf.read()
        pdf_pages_file_1 = pdf_to_image_bytes_list(curr_pdf_file)
        img_bytes = pdf_pages_file_1[0]
        
        curr_result = parse_invoice_bytes(img_bytes)
        f.write("//" + curr_pdf_path + ":\n" + curr_result + "," + "\n")
        print(f"Processed invoice {i+1}/70")
    f.write("]")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/tmp/ipykernel_123/3729595696.py:2: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  return json.dumps(json.loads(result.json())['pages'][0]['extracted_data'], indent=2)
/usr/local/lib/python3.12/site-packages/pydantic/main.py:519: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `tuple[int, int]` - serialized value may not be as expected [input_value=[1, 9223372036854775807], input_type=list])
  return self.__pydantic_serializer__.to_json(


Processed invoice 1/70

Processed invoice 2/70

Processed invoice 3/70

Processed invoice 4/70

...

Processed invoice 66/70

Processed invoice 67/70

Processed invoice 68/70

Processed invoice 69/70

Processed invoice 70/70

Yeah, we have to do manual garbage collection in Python, like in the good old days of C/C++ in college, because this is VRAM and not regular RAM.

In [56]:
# memory de-allocation
del extractor

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")

VRAM free: 21.5 GB